# <span style="color:blue">**Compute your own MZV data mine without extended double shuffle**</span>

Note that what I am showing here is numerical and not really the best way to do it since at high weights, the lattice gets very large and also you need to handle numerical precision carefully. 

The point here is to only illustrate the power of the LLL algorithm.

Assuming Hoffman's conjecutre of weight grading of MZVs i.e. 
$$
\mathcal{Z} = \bigoplus_k \mathcal{Z}_k,
$$
we want to numerically verify Zagier's conjecuture which states that 
$$
\dim_\mathbb{Q} \mathcal{Z}_k = d_k
$$
where $d_k$ is defined as the $k^{th}$ Taylor series coefficients of 
$$
\frac{1}{1-t^2-t^3} = \sum_{n=0}^\infty d_n t^n = 1+t^2+t^3+t^4+2 t^5+2 t^6+3 t^7+4 t^8+5 t^9+7 t^{10}+9 t^{11}+12 t^{12}+16 t^{13}+21 t^{14}+28 t^{15}+O\left(t^{16}\right)
$$

## Step 1: Admissible weight systems at weight $k$

In [11]:
default(parisize, 5g)

  ***   Warning: new stack size = 5000000000 (4768.372 Mbytes).


In [12]:
\\ Generate all ordered partitions (compositions) of n
compositions(n) = {
if (n < 1, return([]));
my(res = List());
\\ Iterate over all 2^(n-1) binary cut configurations
for(m = 0, 2^(n-1) - 1,
my(comp = List(), currval = 1);
for (i = 0, n - 2,
if (bittest(m, i), listput(comp, currval); currval = 1, currval++));
listput(comp, currval);
listput(res, Vec(comp)));
return(Vec(res));
}

#### Now compute the number of admissible multisets

In [13]:
admissible(k) = select(x-> x[1]>1,compositions(k));

#### Check that the number of admissible weight systems at weight $k$ is $2^{k-2}$ for $k \geq 2$.

In [14]:
apply(x-> [x,#admissible(x),2^(x-2)], [2..10])

[[2, 1, 1], [3, 2, 2], [4, 4, 4], [5, 8, 8], [6, 16, 16], [7, 32, 32], [8, 64, 64], [9, 128, 128], [10, 256, 256]]

### Convert admissible weight systems into MZVs

In [15]:
\p 600 \\ Super important! 
wtgrmzv(k) = {
my(mzv = apply(x-> zetamult(x), admissible(k)));
return(mzv);
}

In [5]:
allrelations(x, prec=15)= {
\\  default(realprecision, prec);
my(n=#x, C=10^(prec-10));
my(top = vector(n,i,round(C*x[i])));
my(M = matconcat([Mat(top); matid(n)]));
my(U = qflll(M));
my(R = M * U);
my(cols = matsize(R)[2]);
my(L = List());
for(j=1, cols,if(abs(R[1,j]) < sqrt(C), listput(L, R[2..n+1,j])));
Vec(L);
}

In [16]:
allrelations(x, prec=100) = {
  my(n = #x);
  \\ Don't touch this!
  x = precision(x, prec);
  \\ Scale factor C (use prec - 5 to maximize precision retention)
  my(C = 10^(prec - 5));
  my(top = vector(n, i, round(C * x[i])));
  my(M = matconcat([Mat(top); matid(n)]));
  
  \\ LLL
  \\ Use qflll(M) by default, or qflll(M, 2) for exact rational arithmetic if prec > 200
  my(U = qflll(M));
  my(R = M * U);
  \\ Weed out true relations by testing at higher precision
  my(cols = matsize(R)[2]);
  my(L = List());
  my(threshold = 10^(-(prec - 15))); \\ Residual tolerance
  for(j = 1, cols,
    my(r = R[2..n+1, j]);
    \\ Compute actual numerical residual |r . x|
    if (abs(x * r) < threshold,
      listput(L, r)
    )
  );
  
  Vec(L);
}

### We want to compute the number of admissible multisets - the number of $\mathbb Q$ linear relations. 

In [17]:
check(k) = #admissible(k) - #allrelations(wtgrmzv(k),600);

The following should match with Zagier's conjecture which states that 
$$
\dim_\mathbb{Q} \mathcal{Z}_k = d_k
$$
where $d_k$ is defined as the $k^{th}$ Taylor series coefficients of 
$$
\frac{1}{1-t^2-t^3} = \sum_{n=0}^\infty d_n t^n = 1+t^2+t^3+t^4+2 t^5+2 t^6+3 t^7+4 t^8+5 t^9+7 t^{10}+9 t^{11}+12 t^{12}+16 t^{13}+21 t^{14}+28 t^{15}+O\left(t^{16}\right) ~.
$$

(Let's check up to weight 11)

Run: 

```
for(x = 3,11, print([x, check(x)]))
```
to find

```
[3, 1]
[4, 1]
[5, 2]
[6, 2]
[7, 3]
[8, 4]
[9, 5]
[10, 7]
[11, 9]
```

**Note that this takes a few minutes to run up to 11, so in the interest of time I will do until 9**

In [17]:
for(x = 3,9, print([x, check(x)]))

[3, 1]
[4, 1]
[5, 2]
[6, 2]
[7, 3]
[8, 4]
[9, 5]


Matches!

### $\mathbb Q$-linear relations

In [18]:
qrelations(k,prec=100) = return(allrelations(wtgrmzv(k),prec));

### Representing the MZV relations

In [19]:
fmtmzvrel(v, comps) = {
my(first = 1, s = "");
for(j = 1, #v, my(c = v[j]);
if(c != 0, my(mzv_str = Str("mzv", Vec(comps[j])));
if(first, if(c == 1,  s = mzv_str, c == -1, s = Str("-", mzv_str),
s = Str(c, "*", mzv_str));
first = 0,
if(c == 1,  s = Str(s, " + ", mzv_str),
c == -1, s = Str(s, " - ", mzv_str),
c > 0,  s = Str(s, " + ", c, "*", mzv_str),
s = Str(s, " - ", -c, "*", mzv_str)))));
if(first, return("0 = 0"));
return(Str(s, " = 0"));
}

\\ Display all relations
printmzvrel(data) = {
my(rels = data[1], comps = data[2]);
for(i = 1, #rels, my(v = Vec(rels[i]));
printf("Relation %2d: %s\n", i, fmtmzvrel(v, comps)););
}

\\ Export relations to file
exportmzvrelations(data, filename) = {
my(rels = data[1], comps = data[2]);
for(i = 1, #rels,
my(v = Vec(rels[i]));
my(line = Str("Relation ", i, ": ", fmtmzvrel(v, comps)));
write(filename, line);
);
printf("Exported %d relations to %s\n", #rels, filename);
}

In [20]:
showrel(k) = [qrelations(k), admissible(k)];
\\ dimZk(k) = #showrel(k)[2] - #showrel(k)[1]; \\ Already defined as check(k) above

# <span style="color:red">**MZV relations**</span>

In [20]:
printmzvrel(showrel(3))

Relation  1: -mzv[3] + mzv[2, 1] = 0


In [20]:
printmzvrel(showrel(4))

Relation  1: -mzv[4] + mzv[2, 1, 1] = 0
Relation  2: -mzv[4] + mzv[2, 2] + mzv[3, 1] = 0
Relation  3: mzv[4] - 2*mzv[2, 2] + 2*mzv[3, 1] = 0


In [20]:
printmzvrel(showrel(5))

Relation  1: -mzv[2, 3] + mzv[2, 1, 2] = 0
Relation  2: -mzv[4, 1] + mzv[3, 1, 1] = 0
Relation  3: -mzv[5] + mzv[2, 1, 1, 1] = 0
Relation  4: -mzv[3, 2] + mzv[2, 2, 1] = 0
Relation  5: -mzv[5] + mzv[2, 3] + mzv[3, 2] + mzv[4, 1] = 0
Relation  6: mzv[5] - mzv[2, 3] - mzv[2, 1, 2] + 2*mzv[4, 1] + 2*mzv[3, 1, 1] = 0


In [20]:
printmzvrel(showrel(6))

Relation  1: -mzv[2, 4] + mzv[2, 1, 1, 2] = 0
Relation  2: -mzv[3, 1, 2] + mzv[2, 3, 1] = 0
Relation  3: -mzv[4, 2] + mzv[2, 2, 1, 1] = 0
Relation  4: -mzv[5, 1] + mzv[3, 1, 1, 1] = 0
Relation  5: -mzv[3, 3] + mzv[2, 1, 2, 1] = 0
Relation  6: -mzv[6] + mzv[2, 1, 1, 1, 1] = 0
Relation  7: mzv[2, 4] - mzv[2, 1, 3] - mzv[4, 2] + mzv[3, 2, 1] = 0
Relation  8: mzv[3, 3] - mzv[2, 2, 2] - mzv[5, 1] + mzv[4, 1, 1] = 0
Relation  9: -mzv[6] + mzv[2, 1, 3] + mzv[2, 2, 2] + mzv[3, 1, 2] + mzv[2, 2, 1, 1] + mzv[3, 1, 1, 1] = 0
Relation 10: -mzv[2, 4] + mzv[2, 1, 3] + mzv[3, 1, 2] - mzv[5, 1] + mzv[4, 1, 1] = 0
Relation 11: -mzv[2, 2, 2] + mzv[3, 1, 2] + mzv[2, 3, 1] + mzv[3, 2, 1] = 0
Relation 12: mzv[3, 3] - mzv[2, 1, 3] + mzv[2, 2, 2] + mzv[2, 1, 2, 1] = 0
Relation 13: mzv[6] - mzv[2, 1, 3] + mzv[4, 2] - mzv[2, 1, 1, 2] + mzv[5, 1] + mzv[4, 1, 1] + mzv[2, 2, 1, 1] + mzv[3, 1, 1, 1] = 0
Relation 14: -mzv[3, 3] - mzv[2, 1, 3] + mzv[4, 2] + mzv[2, 1, 1, 2] - mzv[5, 1] + mzv[2, 3, 1] - mzv[4, 1, 1] +

In [20]:
printmzvrel(showrel(7))

Relation  1: -mzv[2, 1, 4] + mzv[2, 1, 1, 3] = 0
Relation  2: -mzv[2, 3, 2] + mzv[2, 2, 1, 2] = 0
Relation  3: -mzv[2, 2, 3] + mzv[2, 1, 2, 2] = 0
Relation  4: -mzv[2, 5] + mzv[2, 1, 1, 1, 2] = 0
Relation  5: -mzv[3, 3, 1] + mzv[3, 1, 2, 1] = 0
Relation  6: -mzv[4, 3] + mzv[2, 3, 2] + mzv[3, 3, 1] = 0
Relation  7: -mzv[2, 3, 2] - mzv[3, 3, 1] + mzv[2, 1, 2, 1, 1] = 0
Relation  8: -mzv[3, 2, 2] + mzv[2, 2, 2, 1] = 0
Relation  9: -mzv[3, 1, 1, 2] + mzv[2, 4, 1] = 0
Relation 10: -mzv[3, 1, 3] + mzv[2, 1, 3, 1] = 0
Relation 11: -mzv[3, 4] + mzv[2, 1, 1, 2, 1] = 0
Relation 12: -mzv[4, 1, 2] + mzv[2, 3, 1, 1] = 0
Relation 13: -mzv[3, 4] + mzv[2, 2, 3] + mzv[3, 1, 3] - mzv[2, 3, 2] + mzv[3, 2, 2] = 0
Relation 14: mzv[3, 1, 3] - mzv[2, 3, 2] + mzv[3, 2, 2] + mzv[4, 1, 2] - mzv[2, 4, 1] - mzv[3, 3, 1] = 0
Relation 15: -mzv[4, 2, 1] + mzv[3, 2, 1, 1] = 0
Relation 16: -mzv[2, 5] + mzv[2, 1, 1, 3] + mzv[2, 2, 1, 2] - mzv[3, 3, 1] + mzv[2, 3, 1, 1] = 0
Relation 17: -mzv[3, 1, 3] - mzv[5, 2] + mzv[2

In [20]:
printmzvrel(showrel(8))

Relation  1: -mzv[2, 1, 5] + mzv[2, 1, 1, 1, 3] = 0
Relation  2: -mzv[2, 2, 1, 3] + mzv[2, 1, 3, 2] = 0
Relation  3: -mzv[2, 3, 3] + mzv[2, 1, 2, 1, 2] = 0
Relation  4: -mzv[2, 2, 4] + mzv[2, 1, 1, 2, 2] = 0
Relation  5: -mzv[2, 4, 2] + mzv[2, 2, 1, 1, 2] = 0
Relation  6: -mzv[2, 6] + mzv[2, 1, 1, 1, 1, 2] = 0
Relation  7: -mzv[3, 2, 3] + mzv[2, 1, 2, 2, 1] = 0
Relation  8: -mzv[3, 1, 4] + mzv[2, 1, 1, 3, 1] = 0
Relation  9: -mzv[3, 3, 2] + mzv[2, 2, 1, 2, 1] = 0
Relation 10: -mzv[3, 1, 1, 3] + mzv[2, 1, 4, 1] = 0
Relation 11: -mzv[3, 2, 1, 2] + mzv[2, 3, 2, 1] = 0
Relation 12: -mzv[3, 1, 2, 2] + mzv[2, 2, 3, 1] = 0
Relation 13: -mzv[3, 1, 1, 1, 2] + mzv[2, 5, 1] = 0
Relation 14: -mzv[3, 4, 1] + mzv[3, 1, 1, 2, 1] = 0
Relation 15: -mzv[5, 3] + mzv[2, 1, 2, 1, 1, 1] = 0
Relation 16: -mzv[4, 4] + mzv[2, 1, 1, 2, 1, 1] = 0
Relation 17: -mzv[4, 1, 3] + mzv[2, 1, 3, 1, 1] = 0
Relation 18: -mzv[4, 2, 2] + mzv[2, 2, 2, 1, 1] = 0
Relation 19: -mzv[4, 3, 1] + mzv[3, 1, 2, 1, 1] = 0
Relation 20:

In [20]:
printmzvrel(showrel(9))

Relation  1: -mzv[2, 1, 1, 5] + mzv[2, 1, 1, 1, 4] = 0
Relation  2: -mzv[2, 1, 6] + mzv[2, 1, 1, 1, 1, 3] = 0
Relation  3: -mzv[2, 1, 3, 3] + mzv[2, 1, 2, 1, 3] = 0
Relation  4: -mzv[2, 1, 2, 4] + mzv[2, 1, 1, 2, 3] = 0
Relation  5: -mzv[2, 2, 1, 4] + mzv[2, 1, 1, 3, 2] = 0
Relation  6: -mzv[2, 2, 1, 1, 3] + mzv[2, 1, 4, 2] = 0
Relation  7: -mzv[2, 2, 2, 3] + mzv[2, 1, 2, 2, 2] = 0
Relation  8: -mzv[2, 2, 3, 2] + mzv[2, 2, 1, 2, 2] = 0
Relation  9: -mzv[2, 3, 2, 2] + mzv[2, 2, 2, 1, 2] = 0
Relation 10: -mzv[2, 2, 5] + mzv[2, 1, 1, 1, 2, 2] = 0
Relation 11: -mzv[2, 3, 4] + mzv[2, 1, 1, 2, 1, 2] = 0
Relation 12: -mzv[2, 4, 3] + mzv[2, 1, 2, 1, 1, 2] = 0
Relation 13: -mzv[2, 3, 1, 3] + mzv[2, 1, 3, 1, 2] = 0
Relation 14: -mzv[2, 4, 1, 2] + mzv[2, 3, 1, 1, 2] = 0
Relation 15: -mzv[2, 5, 2] + mzv[2, 2, 1, 1, 1, 2] = 0
Relation 16: -mzv[2, 7] + mzv[2, 1, 1, 1, 1, 1, 2] = 0
Relation 17: -mzv[3, 3, 3] + mzv[2, 1, 2, 1, 2, 1] = 0
Relation 18: -mzv[3, 2, 2, 2] + mzv[2, 2, 2, 2, 1] = 0
Relation 1

[IPKernelApp] WARNING | Unknown message type: 'comm_open'
[IPKernelApp] WARNING | Unknown message type: 'comm_msg'
